# 04 · Silver periods of service
`bronze.per_periods_of_service` → `silver.hcmperiodsofservice` (module 06 §3).

✅ Expect **2,688** rows · 688 terminated (Voluntary 492 / Involuntary 196) · 5 quarantined.

In [ ]:
from pyspark.sql import functions as F
from hcm_utils import load_config, parse_date, latest_batch, issue_rows, write_issues, upsert, quarantine

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("run_id", "manual")
catalog = dbutils.widgets.get("catalog")
run_id  = dbutils.widgets.get("run_id")
cfg = load_config()

In [ ]:
SRC_FILE = "PER_PERIODS_OF_SERVICE"
COLS = ["PERSON_ID", "PERIOD_OF_SERVICE_ID", "DATE_START", "ACTUAL_TERMINATION_DATE", "LEAVING_REASON"]
as_of = cfg["snapshot"]["end_date"]
reasons = {k: v for k, v in cfg["leaving_reason_map"].items() if k != "_default"}
default_type = cfg["leaving_reason_map"].get("_default", "Unknown")
reason_map = F.create_map(*[F.lit(x) for kv in reasons.items() for x in kv])

df = (latest_batch(f"{catalog}.bronze.per_periods_of_service").select(*COLS)
      .withColumn("START", parse_date("DATE_START"))
      .withColumn("TERM", parse_date("ACTUAL_TERMINATION_DATE")))

bad = df.filter("TERM < START")
issues = [issue_rows(bad, SRC_FILE, "PERSON_ID", "CONSISTENCY_VIOLATION",
                     "ACTUAL_TERMINATION_DATE precedes DATE_START", "QUARANTINED")]
quarantine(bad.select(*COLS), SRC_FILE, "PERSON_ID", "ACTUAL_TERMINATION_DATE precedes DATE_START", catalog, run_id)

In [ ]:
good = df.filter("TERM IS NULL OR TERM >= START")
silver = good.select(
    F.col("PERIOD_OF_SERVICE_ID").cast("bigint").alias("PeriodOfServiceId"),
    F.col("PERSON_ID").cast("bigint").alias("PersonId"),
    F.col("START").alias("DateStart"),
    F.col("TERM").alias("ActualTerminationDate"),
    F.trim("LEAVING_REASON").alias("LeavingReason"),
    F.when(F.col("TERM").isNull(), F.lit(None).cast("string"))
     .otherwise(F.coalesce(F.try_element_at(reason_map, F.trim("LEAVING_REASON")), F.lit(default_type)))
     .alias("TerminationType"),
    F.round(F.months_between(F.coalesce(F.col("TERM"), F.lit(as_of).cast("date")), F.col("START")), 1)
     .alias("TenureMonths"),
    F.col("TERM").isNotNull().alias("IsTerminated"),
    F.lit(run_id).alias("RunId"),
    F.current_timestamp().alias("SilverLoadTimestamp"),
)
upsert(silver, f"{catalog}.silver.hcmperiodsofservice", keys=["PeriodOfServiceId"])
print("issues logged:", write_issues(issues, catalog, run_id))
print("hcmperiodsofservice rows:", spark.table(f"{catalog}.silver.hcmperiodsofservice").count())